In [70]:
import pandas as pd
from sklearn.cluster import AgglomerativeClustering,k_means
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
from sklearn.preprocessing import LabelEncoder,OneHotEncoder
from mlxtend.frequent_patterns import apriori, association_rules

import warnings
warnings.filterwarnings("ignore")

### Data collection

In [8]:
%pip install openpyxl

retail_data = pd.read_excel("Online Retail.xlsx", engine="openpyxl")
retail_data


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


### Data understanding

In [10]:
retail_data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [14]:
retail_data.describe()

,Quantity,InvoiceDate,UnitPrice,CustomerID
count,541909.000000,541909,541909.000000,406829.000000
mean,9.552250,2011-07-04 13:34:57.156386,4.611114,15287.690570
min,-80995.000000,2010-12-01 08:26:00,-11062.060000,12346.000000
25%,1.000000,2011-03-28 11:34:00,1.250000,13953.000000
50%,3.000000,2011-07-19 17:17:00,2.080000,15152.000000
75%,10.000000,2011-10-19 11:27:00,4.130000,16791.000000
max,80995.000000,2011-12-09 12:50:00,38970.000000,18287.000000
std,218.081158,NaN,96.759853,1713.600303


In [15]:
retail_data.drop_duplicates()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
...,...,...,...,...,...,...,...,...
541904,581587,22613,PACK OF 20 SPACEBOY NAPKINS,12,2011-12-09 12:50:00,0.85,12680.0,France
541905,581587,22899,CHILDREN'S APRON DOLLY GIRL,6,2011-12-09 12:50:00,2.10,12680.0,France
541906,581587,23254,CHILDRENS CUTLERY DOLLY GIRL,4,2011-12-09 12:50:00,4.15,12680.0,France
541907,581587,23255,CHILDRENS CUTLERY CIRCUS PARADE,4,2011-12-09 12:50:00,4.15,12680.0,France


In [16]:
retail_data.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [19]:
retail_data.drop(columns="CustomerID",inplace= True)

In [20]:
retail_data.isna().sum()

InvoiceNo         0
StockCode         0
Description    1454
Quantity          0
InvoiceDate       0
UnitPrice         0
Country           0
dtype: int64

In [22]:
retail_data.dropna(inplace=True)

In [24]:
retail_data.isna().sum()

InvoiceNo      0
StockCode      0
Description    0
Quantity       0
InvoiceDate    0
UnitPrice      0
Country        0
dtype: int64

### Business Needs and understanding

In [27]:
retail_data["Country"].value_counts()

Country
United Kingdom          494024
Germany                   9495
France                    8557
EIRE                      8196
Spain                     2533
Netherlands               2371
Belgium                   2069
Switzerland               2002
Portugal                  1519
Australia                 1259
Norway                    1086
Italy                      803
Channel Islands            758
Finland                    695
Cyprus                     622
Sweden                     462
Unspecified                446
Austria                    401
Denmark                    389
Japan                      358
Poland                     341
Israel                     297
USA                        291
Hong Kong                  288
Singapore                  229
Iceland                    182
Canada                     151
Greece                     146
Malta                      127
United Arab Emirates        68
European Community          61
RSA                         58


In [32]:
Germany_data = retail_data[retail_data["Country"] == "Germany"]
Germany_data

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,Country
1109,536527,22809,SET OF 6 T-LIGHTS SANTA,6,2010-12-01 13:04:00,2.95,Germany
1110,536527,84347,ROTATING SILVER ANGELS T-LIGHT HLDR,6,2010-12-01 13:04:00,2.55,Germany
1111,536527,84945,MULTI COLOUR SILVER T-LIGHT HOLDER,12,2010-12-01 13:04:00,0.85,Germany
1112,536527,22242,5 HOOK HANGER MAGIC TOADSTOOL,12,2010-12-01 13:04:00,1.65,Germany
1113,536527,22244,3 HOOK HANGER MAGIC GARDEN,12,2010-12-01 13:04:00,1.95,Germany
...,...,...,...,...,...,...,...
541801,581578,22993,SET OF 4 PANTRY JELLY MOULDS,12,2011-12-09 12:16:00,1.25,Germany
541802,581578,22907,PACK OF 20 NAPKINS PANTRY DESIGN,12,2011-12-09 12:16:00,0.85,Germany
541803,581578,22908,PACK OF 20 NAPKINS RED APPLES,12,2011-12-09 12:16:00,0.85,Germany
541804,581578,23215,JINGLE BELL HEART ANTIQUE SILVER,12,2011-12-09 12:16:00,2.08,Germany


In [42]:
Germany_data.groupby("Description")["Quantity"].sum().sort_values(ascending = False)

Description
ROUND SNACK BOXES SET OF4 WOODLAND     1218
ASSORTED COLOURS SILK FAN              1164
POSTAGE                                1104
WOODLAND CHARLOTTE BAG                 1019
PACK OF 72 RETROSPOT CAKE CASES        1002
                                       ... 
TRIPLE PHOTO FRAME CORNICE               -2
GROW YOUR OWN PLANT IN A CAN             -3
BLUE PADDED SOFT MOBILE                  -3
SET OF 3 BABUSHKA STACKING TINS          -6
TRAVEL CARD WALLET KEEP CALM            -20
Name: Quantity, Length: 1703, dtype: int64

In [46]:
Germany_data.groupby("Description")[["Quantity","UnitPrice"]].sum().sort_values(by="Quantity", ascending=False)

,Quantity,UnitPrice
Description,,
ROUND SNACK BOXES SET OF4 WOODLAND,1218,353.20
ASSORTED COLOURS SILK FAN,1164,7.13
POSTAGE,1104,7843.00
WOODLAND CHARLOTTE BAG,1019,50.02
PACK OF 72 RETROSPOT CAKE CASES,1002,21.45
...,...,...
TRIPLE PHOTO FRAME CORNICE,-2,9.95
GROW YOUR OWN PLANT IN A CAN,-3,1.25
BLUE PADDED SOFT MOBILE,-3,4.25


### Data preprocessing

In [48]:
onehot = OneHotEncoder()

In [50]:
Germany_data.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
Country                   str
dtype: object

In [54]:
Germany_product_list = Germany_data['Description'].value_counts()

In [60]:
Germany_pivot_data= pd.pivot_table(data = Germany_data, values = 'Quantity', index =  'InvoiceNo',columns= "Description",aggfunc = sum).fillna(0)

In [ ]:
Germany_pivot_data = Germany_pivot_data.map(lambda x:0 if x==0 else 1)

In [63]:
Germany_pivot_data.describe()

Description,50'S CHRISTMAS GIFT BAG LARGE,DOLLY GIRL BEAKER,I LOVE LONDON MINI BACKPACK,RED SPOT GIFT BAG LARGE,SET 2 TEA TOWELS I LOVE LONDON,SPACEBOY BABY GIFT SET,10 COLOUR SPACEBOY PEN,12 COLOURED PARTY BALLOONS,12 IVORY ROSE PEG PLACE SETTINGS,12 MESSAGE CARDS WITH ENVELOPES,12 PENCIL SMALL TUBE WOODLAND,12 PENCILS SMALL TUBE RED RETROSPOT,12 PENCILS SMALL TUBE SKULL,12 PENCILS TALL TUBE POSY,12 PENCILS TALL TUBE RED RETROSPOT,12 PENCILS TALL TUBE SKULLS,12 PENCILS TALL TUBE WOODLAND,12 PINK HEN+CHICKS IN BASKET,12 RED ROSE PEG PLACE SETTINGS,16 PIECE CUTLERY SET PANTRY DESIGN,2 PICTURE BOOK EGGS EASTER BUNNY,2 PICTURE BOOK EGGS EASTER CHICKS,20 DOLLY PEGS RETROSPOT,200 BENDY SKULL STRAWS,200 RED + WHITE BENDY STRAWS,3 DRAWER ANTIQUE WHITE WOOD CABINET,3 HOOK HANGER MAGIC GARDEN,3 HOOK PHOTO SHELF ANTIQUE WHITE,3 PIECE SPACEBOY COOKIE CUTTER SET,3 RAFFIA RIBBONS 50'S CHRISTMAS,3 STRIPEY MICE FELTCRAFT,3 TIER CAKE TIN GREEN AND CREAM,3 TIER CAKE TIN RED AND CREAM,3 TIER SWEETHEART GARDEN SHELF,3 TRADITIONAl BISCUIT CUTTERS SET,36 DOILIES DOLLY GIRL,36 FOIL HEART CAKE CASES,36 FOIL STAR CAKE CASES,36 PENCILS TUBE RED RETROSPOT,36 PENCILS TUBE SKULLS,...,WRAP BAD HAIR DAY,WRAP BILLBOARD FONTS DESIGN,WRAP BIRD GARDEN,WRAP CAROUSEL,WRAP CHRISTMAS SCREEN PRINT,WRAP CIRCUS PARADE,WRAP COWBOYS,WRAP DOILEY DESIGN,WRAP DOLLY GIRL,WRAP ENGLISH ROSE,WRAP FLOWER SHOP,WRAP FOLK ART,WRAP GINGHAM ROSE,WRAP GREEN PEARS,WRAP I LOVE LONDON,WRAP MAGIC FOREST,WRAP MONSTER FUN,WRAP PAISLEY PARK,WRAP PINK FAIRY CAKES,WRAP POPPIES DESIGN,WRAP RED APPLES,WRAP RED DOILEY,WRAP RED VINTAGE DOILY,WRAP SUKI AND FRIENDS,WRAP VINTAGE LEAF DESIGN,WRAP VINTAGE PETALS DESIGN,WRAP WEDDING DAY,"WRAP, BILLBOARD FONTS DESIGN",YELLOW COAT RACK PARIS FASHION,YOU'RE CONFUSING ME METAL SIGN,YULETIDE IMAGES GIFT WRAP SET,ZINC HEART T-LIGHT HOLDER,ZINC STAR T-LIGHT HOLDER,ZINC BOX SIGN HOME,ZINC FOLKART SLEIGH BELLS,ZINC HEART LATTICE T-LIGHT HOLDER,ZINC METAL HEART DECORATION,ZINC T-LIGHT HOLDER STAR LARGE,ZINC T-LIGHT HOLDER STARS SMALL,ZINC WILLIE WINKIE CANDLE STICK
count,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,...,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000,603.000000
mean,0.008292,0.011609,0.003317,0.004975,0.008292,0.008292,0.018242,0.003317,0.001658,0.003317,0.016584,0.009950,0.006633,0.003317,0.006633,0.003317,0.016584,0.003317,0.001658,0.004975,0.001658,0.001658,0.006633,0.003317,0.001658,0.003317,0.029851,0.013267,0.033167,0.004975,0.008292,0.001658,0.001658,0.001658,0.006633,0.004975,0.009950,0.003317,0.023217,0.006633,...,0.004975,0.001658,0.003317,0.001658,0.001658,0.013267,0.006633,0.013267,0.004975,0.004975,0.003317,0.001658,0.003317,0.008292,0.003317,0.001658,0.003317,0.004975,0.006633,0.011609,0.033167,0.004975,0.009950,0.009950,0.006633,0.006633,0.004975,0.001658,0.001658,0.001658,0.001658,0.001658,0.001658,0.001658,0.003317,0.001658,0.001658,0.003317,0.003317,0.006633
std,0.090757,0.107205,0.057543,0.070417,0.090757,0.090757,0.133937,0.057543,0.040723,0.057543,0.127812,0.099336,0.081243,0.057543,0.081243,0.057543,0.127812,0.057543,0.040723,0.070417,0.040723,0.040723,0.081243,0.057543,0.040723,0.057543,0.170317,0.114511,0.179222,0.070417,0.090757,0.040723,0.040723,0.040723,0.08

In [68]:
# Binary matrix is ready for association rules
# Convert to numpy array if needed
Germany_pivot_array =  Germany_pivot_data.values

In [93]:
apriori_value = apriori(Germany_pivot_data,min_support = 0.02,use_colnames=True)
apriori_value

,support,itemsets
0,0.029851,frozenset({3 HOOK HANGER MAGIC GARDEN})
1,0.033167,frozenset({3 PIECE SPACEBOY COOKIE CUTTER SET})
2,0.023217,frozenset({36 PENCILS TUBE RED RETROSPOT})
3,0.021559,frozenset({36 PENCILS TUBE WOODLAND})
4,0.029851,frozenset({5 HOOK HANGER RED MAGIC TOADSTOOL})
...,...,...
492,0.023217,"frozenset({PLASTERS IN TIN SPACEBOY, ROUND SNA..."
493,0.029851,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS ,..."
494,0.021559,"frozenset({WOODLAND CHARLOTTE BAG, ROUND SNACK..."
495,0.026534,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS ,..."


In [97]:
association_rules(apriori_value, metric = "confidence",min_threshold = 0.8)

,antecedents,consequents,antecedent support,consequent support,support,confidence,lift,representativity,leverage,conviction,zhangs_metric,jaccard,certainty,kulczynski
0,frozenset({6 RIBBONS RUSTIC CHARM}),frozenset({POSTAGE}),0.082919,0.635158,0.069652,0.840000,1.322507,1.0,0.016985,2.280265,0.265909,0.107417,0.561455,0.474830
1,frozenset({ASSORTED COLOUR MINI CASES}),frozenset({POSTAGE}),0.026534,0.635158,0.023217,0.875000,1.377611,1.0,0.006364,2.918740,0.281577,0.036364,0.657386,0.455777
2,frozenset({BAKING SET 9 PIECE RETROSPOT }),frozenset({POSTAGE}),0.038143,0.635158,0.033167,0.869565,1.369054,1.0,0.008941,2.797125,0.280259,0.051813,0.642490,0.460892
3,frozenset({BIG DOUGHNUT FRIDGE MAGNETS}),frozenset({POSTAGE}),0.024876,0.635158,0.021559,0.866667,1.364491,1.0,0.005759,2.736318,0.273940,0.033766,0.634545,0.450305
4,frozenset({BISCUIT TIN 50'S CHRISTMAS}),frozenset({POSTAGE}),0.024876,0.635158,0.023217,0.933333,1.469452,1.0,0.007417,5.472637,0.327624,0.036458,0.817273,0.484943
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
183,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS ,...",frozenset({ROUND SNACK BOXES SET OF4 WOODLAND }),0.028192,0.197347,0.026534,0.941176,4.769155,1.0,0.020970,13.645108,0.813247,0.133333,0.926714,0.537815
184,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS ,...",frozenset({POSTAGE}),0.029851,0.635158,0.026534,0.888889,1.399478,1.0,0.007574,3.283582,0.294231,0.041558,0.695455,0.465332
185,"frozenset({ROUND SNACK BOXES SET OF 4 FRUITS ,...",frozenset({ROUND SNACK BOXES SET OF4 WOODLAND ...,0.031509,0.170813,0.026534,0.842105,4.929995,1.0,0.021152,5.251520,0.823095,0.150943,0.809579,0.498723
186,"frozenset({WOODLAND CHARLOTTE BAG, ROUND SNACK...",frozenset({POSTAGE}),0.029851,0.635158,0.026534,0.888889,1.399478,1.0,0.007574,3.283582,0.294231,0.041558,0.695455,0.465332
